# 🔒 SGLang 约束生成 — FSM-based Structured Decoding

**本文目标**：深入理解 SGLang 的约束生成机制——如何保证 LLM 输出 100% 符合 JSON Schema/Regex。

读完这篇你会理解：
- FSM (有限状态机) 在 token-level 的工作方式
- 约束生成如何与 RadixAttention 形成组合优势
- 工具调用中约束生成的实际收益 (减少 retry, 提高 cache 命中)
- 性能开销: 约束生成的计算 cost 是多少

## 1. 问题: 自由生成的不可靠性

```
# 期望: LLM 输出一个合法的 JSON function call
{
  "name": "get_weather",
  "parameters": {"city": "Beijing"}
}

# 实际可能的输出 (自由生成):
{"name": "get_weather", "parameters": {"city": "Beijing"}}   ✓ 合法
{"name": "get_weather", "params": {"city": "Beijing"}}      ✗ key 错误
{"function": "get_weather", "args": {"city": "Beijing"}}    ✗ 格式错误
{"name": "get_weather", "parameters": {"city": Beijing}}    ✗ value 没用引号
{name: "get_weather", parameters: {city: "Beijing"}}         ✗ key 没有引号

# 即使模型很强 (GPT-4 level), 在长对话和复杂 schema 时也会偶尔出错
# 对于 Agent 密集型应用 → 1% 的 JSON 错误率 → 需要 retry 逻辑 → 复杂且慢
```

## 2. FSM (有限状态机) 约束生成

### 2.1 基本思想

```
给定 JSON Schema:
{
  "type": "object",
  "properties": {
    "name": {"type": "string", "enum": ["get_weather", "get_time"]},
    "parameters": {
      "type": "object",
      "properties": {"city": {"type": "string"}}
    }
  }
}

→ 编译为 FSM:

State 0: 必须输出 '{'
State 1: 必须输出 '"name"'
State 2: 必须输出 ':'
State 3: 必须在 {"get_weather", "get_time"} 中选择
State 4: 必须输出 ','
State 5: 必须输出 '"parameters"'
...

在每个 state, 只有 "合法" 的 token 能被生成
→ softmax 后把非法 token 的 prob 设 为 0
→ 保证每一步都在 Schema 约束内
```

### 2.2 实现: Token-level Mask

```python
class FSMLogitsProcessor:
    """在每个 decode step 应用 FSM 约束"""
    
    def __init__(self, schema):
        self.fsm = compile_schema_to_fsm(schema)
        self.state = self.fsm.initial_state
    
    def __call__(self, token_ids, logits):
        """修改 logits: 非法 token → -inf"""
        allowed_tokens = self.fsm.allowed_tokens(self.state)
        
        # 创建一个全 -inf 的 mask
        mask = torch.full_like(logits, float('-inf'))
        # 只保留合法 token 的原始 logit
        mask[allowed_tokens] = logits[allowed_tokens]
        
        return mask

# 使用:
# sampler = FSMLogitsProcessor(schema)
# for step in range(max_tokens):
#     logits = model.forward(input_ids)
#     constrained_logits = sampler(input_ids, logits)
#     next_token = sample(constrained_logits)
#     sampler.state = sampler.fsm.transition(sampler.state, next_token)
```

### 2.3 预编译的 FSM

SGLang 优化：对常见的 Schema (如 OpenAI function calling format) 预编译 FSM：

```python
# SGLang 内置的常见 FSM
_schema_cache = {
    "openai_tool_call": compile_openai_tool_fsm(),
    "json_object": compile_json_fsm(),
    "chat_format": compile_chat_fsm(),
}

# 对于自定义 Schema → 运行时编译 (有一次性开销)
# 编译后的 FSM 可以缓存和复用
```

### 2.4 约束生成 + RadixAttention 的组合优势

```
这是 SGLang 在 Agent 场景最大的 moat:

1. Constrained generation → tool_call 格式 100% 一致
   → 不同请求的 tool_call token 序列高度相似
   → RadixAttention 的 cache 命中率提高

示例:
  无约束: "{"name":  "get_weather","parameters":  {"city":"Beijing"}}"
  有约束: '{"name":"get_weather","parameters":{"city":"Beijing"}}'
          ↑ 空格数量完全一致, 引号使用完全一致

2. 工具调用不需要 retry
   → 不浪费 decode steps → 更少的 KV Cache 消耗

3. 对于批量处理任务 (如从 1000 个文档中提取信息)
   → 输出格式完全一致 → RadixAttention 可能共享部分输出 token!
```

## 3. 约束生成的类型

### 3.1 JSON Schema 约束

```python
import sglang as sgl

@sgl.function
def extract_info(s, text):
    s += sgl.user(f"Extract from: {text}")
    s += sgl.assistant(
        sgl.gen("result", max_tokens=200, 
                json_schema={
                    "type": "object",
                    "properties": {
                        "name": {"type": "string"},
                        "age": {"type": "integer"},
                        "email": {"type": "string", "format": "email"}
                    },
                    "required": ["name", "age"]
                })
    )

# 输出 100% 合法 JSON, 包含 name 和 age 字段
```

### 3.2 Regex 约束

```python
@sgl.function
def classify_sentiment(s, text):
    s += sgl.user(f"Classify: {text}")
    s += sgl.assistant(
        sgl.gen("sentiment", max_tokens=10,
                regex=r"(positive|negative|neutral)")
    )
# 输出只能是 "positive", "negative", "neutral" 之一

# 更复杂的 Regex:
regex = r"\{"name": "(get_weather|search|calculate)", "args": \{.*\}\}"
```

### 3.3 Choice 约束 (Select)

```python
@sgl.function
def answer_mc(s, question, options):
    s += sgl.user(f"{question}\nOptions: {options}")
    s += sgl.assistant(
        sgl.select("answer", choices=options)
    )
# 输出只能是 options 中的一个 → 最快, 最简单的约束
```

## 4. 实操: 约束生成的性能分析

### 4.1 FSM 的 overhead

```
FSM 编译开销:
  简单 JSON Schema (~5 个字段): < 1ms
  复杂 JSON Schema (~50 个字段): ~5-10ms
  → 编译一次, 可缓存复用

每 step 的 FSM 开销:
  计算 allowed_tokens: ~0.01-0.1ms
  应用 mask: 可忽略 (只是 tensor indexing)
  → 占总 decode 时间的 < 1%

额外收益:
  减少 retry: 1% 错误率 × 平均 3s retry → 节省 ~30ms/请求
  对于高 QPS 服务 → 可能比 FSM 开销更划算
```

### 4.2 对 Cache 命中率的提升

```
实验: 100 个信息提取请求, 有/无约束生成

无约束:
  输出格式不一 → 后续 token 几乎无共享 → cache hit ~5%

有约束:
  输出格式固定 → 后续 token 可能有部分共享 → cache hit ~10%
  
  如果 100 个请求提取相同的 schema → 输出的结构部分高度一致
  → RadixAttention 可能命中 "前缀" (如 '{"name":"' 等固定部分)
  → cache hit ~15-20%
```

In [ ]:
# 约束生成的 token mask 演示

# 模拟一个简化的 FSM
class MiniFSM:
    def __init__(self, schema_type="json_object"):
        self.state = 0
        self.transitions = {
            0: {"{": 1},                    # 开始
            1: {'"': 2},                    # key 开始
            2: {"abcdefghijklmnopqrstuvwxyz"},  # key 内容 (简化)
            3: {'"': 4},                    # key 结束
            4: {":": 5},                    # 冒号
            5: {'"': 6, "0123456789": 8, "t": 7, "f": 7, "n": 7},  # value 开始
            6: {"all"},                     # string value
            7: {"all"},                     # true/false/null
            8: {"0123456789"},             # number
        }
    
    def allowed(self):
        """返回当前状态允许的字符集"""
        return self.transitions.get(self.state, {"all"})
    
    def step(self, char):
        """状态转移"""
        allowed = self.allowed()
        if "all" in allowed or char in allowed:
            self.state += 1 if self.state < 8 else 0
            return True
        return False

# 测试
fsm = MiniFSM()
test_strings = [
    '{"name":"Alice","age":30}',
    '{name:"Alice",age:30}',        # 缺少引号 → 失败
    '{"name":"Alice","age":thirty}' # age 不是数字 → 失败
]

for s in test_strings:
    fsm.state = 0
    ok = all(fsm.step(c) for c in s)
    status = "OK" if ok else "FAIL at state " + str(fsm.state)
    print(f"  {s:40s} → {status}")
print()
print("约束生成保证: 只有合法 JSON 能被生成")
print("非法 token → logit = -inf → 永远不会被采样")